Please note that this is just a demonstration of the data_scraper
In order to perform this scraping, we need to get the API Key from the Semantics Scholar website.
Link to the S2 API Request form: https://www.semanticscholar.org/product/api#api-key-form
Link to Kaggle: https://www.kaggle.com/code/akuseru59/data-scraper  

In [ ]:
!pip install arxiv --quiet

In [ ]:
import arxiv
import requests
import json
import time
import calendar
from tqdm import tqdm

API_KEY = "xqKzmjE12j3gzl7MyAcyAkCPmDEmUu7axaHwubGc"
S2_BATCH_URL = "https://api.semanticscholar.org/graph/v1/paper/batch"
FIELDS = "paperId,externalIds,title,authors,abstract,publicationDate"
PAPERS_PER_YEAR = 40000
PAPERS_PER_MONTH = PAPERS_PER_YEAR // 12

In [ ]:
client = arxiv.Client(page_size=100, delay_seconds=3, num_retries=10)

def fetch_year(year):
    year_data = []
    short_year = str(year)[2:]
    
    target_months = range(1, 13)
    if year == 2026:
        target_months = range(1, 5) 
        
    for month in target_months:
        print(f"\n--- Harvesting {year}-{month:02} ---")
        
        # 1. Generate the 5-digit IDs (e.g., 1901.00001)
        ids_to_try = [f"{short_year}{month:02}.{i:05}" for i in range(1, PAPERS_PER_MONTH + 1)]
        
        # 2. Fetch from arXiv in chunks of 100
        valid_arxiv_ids = []
        for k in tqdm(range(0, len(ids_to_try), 100), desc="arXiv Lookup"):
            chunk = ids_to_try[k : k + 100]
            
            # We use id_list specifically to pull our generated IDs
            search = arxiv.Search(id_list=chunk)
            
            try:
                # Use client.results as requested
                for result in client.results(search):
                    # Extract the ID without the 'v1' suffix for Semantic Scholar
                    valid_arxiv_ids.append(result.entry_id.split('/')[-1].split('v')[0])
            except Exception as e:
                if "429" in str(e):
                    time.sleep(60)
                continue

        if not valid_arxiv_ids:
            print(f"No papers found for {year}-{month:02}. Check connection.")
            continue

        # 3. Enrich with Semantic Scholar in batches of 500
        month_published = []
        for j in tqdm(range(0, len(valid_arxiv_ids), 500), desc="S2 Enrichment"):
            s2_chunk = valid_arxiv_ids[j : j + 500]
            payload = {"ids": [f"ArXiv:{aid}" for aid in s2_chunk]}
            
            try:
                res = requests.post(
                    S2_BATCH_URL, 
                    json=payload, 
                    params={"fields": FIELDS}, 
                    headers={"x-api-key": API_KEY},
                    timeout=20
                )
                
                if res.status_code == 200:
                    batch_data = res.json()
                    # FILTER: Keep only papers published in a venue that isn't 'arXiv'
                    for paper in batch_data:
                        if paper:
                            # If there's a venue, use it. If not, label it 'arXiv' or 'Unknown'
                            v = paper.get("venue")
                            paper["official_venue"] = v if (v and v.lower() != "arxiv") else "arXiv/Preprint"
                            month_published.append(paper)
                
                time.sleep(1.1) # Essential S2 delay
            except Exception as e:
                print(f"S2 Error: {e}")
                continue
        
        year_data.extend(month_published)
        print(f"Success: {year}-{month:02} | ArXiv: {len(valid_arxiv_ids)} | Published: {len(month_published)}")

    # Save the year's data
    out_file = f"arxiv_published_{year}.json"
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(year_data, f, indent=2, ensure_ascii=False)

if __name__ == "__main__":
    for y in range(2019, 2027):
        fetch_year(y)